In [3]:
import polars as pl
from pl_compare import compare

is het beter om de data direct in te laden (polars kan dat) of eerst omzetten in een file?
>> volgens dagster kan je het beste meteen inladen in de database (als files niet veel te groot zijn)
>> kijk naar: Direct Loading with I/O Managers: https://docs.dagster.io/guides/build/io-managers

todo:
    
done:
    name.basics
    title.basics
    title.principals
    title.crew (split and add directors and writers from title.principals and remove duplicates)

## transform title.basics

In [2]:
test = pl.read_csv(
    "/media/user/Data/dirkv/Code/dagster/dagster-workspace/data/imdb/inputs/imdb_files/title.basics.tsv.gz",
    has_header=True,
    separator="\t",
    truncate_ragged_lines=True,
    null_values="\\N",
    quote_char=None,
    schema={
        "tconst": pl.Utf8,
        "titleType": pl.Utf8,
        "primaryTitle": pl.Utf8,
        "originalTitle": pl.Utf8,
        "isAdult": pl.Int64,
        "startYear": pl.Int64,
        "endYear": pl.Int64,
        "runtimeMinutes": pl.Int64,
        "genres": pl.Utf8,
    }
)
title_basics = test.drop("genres") # de 0 bij is adult zou al goed geladen moeten worden door postgresl
genres = test.select(
    pl.col("tconst"), 
    pl.col("genres").str.split(",")
    ).explode("genres")

# transforming name.basics

In [8]:
test = pl.read_csv(
    "/media/user/Data/dirkv/Code/dagster/dagster-workspace/data/imdb/inputs/imdb_files/name.basics.tsv.gz",
    has_header=True,
    separator="\t",
    truncate_ragged_lines=True,
    null_values="\\N",
    quote_char=None,
    schema={
        "nconst": pl.Utf8,
        "primaryName": pl.Utf8,
        "birthYear": pl.Int16,
        "deathYear": pl.Int16,
        "primaryProfession": pl.Utf8,
        "knownForTitles": pl.Utf8,
        
    }
)
# primary_profession = test.select(
#     pl.col("nconst"),
#     pl.col("primaryProfession").str.split(",")
# ).explode("primaryProfession")

# known_for_titles = test.select(
#     pl.col("nconst"),
#     pl.col("knownForTitles").str.split(",")
# ).explode("knownForTitles")
name_basics = test.drop(["primaryProfession", "knownForTitles"])

In [9]:
name_basics

nconst,primaryName,birthYear,deathYear
str,str,i16,i16
"""nm0000001""","""Fred Astaire""",1899,1987
"""nm0000002""","""Lauren Bacall""",1924,2014
"""nm0000003""","""Brigitte Bardot""",1934,2025
"""nm0000004""","""John Belushi""",1949,1982
"""nm0000005""","""Ingmar Bergman""",1918,2007
…,…,…,…
"""nm9993714""","""Romeo del Rosario""",null,null
"""nm9993716""","""Essias Loberg""",null,null
"""nm9993717""","""Harikrishnan Rajan""",null,null


# transforming title.principals

In [3]:
test = pl.read_csv(
    "/media/user/Data/dirkv/Code/dagster/dagster-workspace/data/imdb/inputs/imdb_files/title.principals.tsv.gz",
    has_header=True,
    separator="\t",
    truncate_ragged_lines=True,
    null_values="\\N",
    quote_char=None,
    schema={
        "tconst": pl.Utf8,
        "ordering": pl.Int16,
        "nconst": pl.Utf8,
        "category": pl.Utf8,
        "job": pl.Utf8,
        "characters": pl.Utf8
    }
)
title_principals = test.with_columns(
    pl.col("characters").str.replace_all(r'[\[\]"]', "")
)
# we leave job for wat it is
title_principals

tconst,ordering,nconst,category,job,characters
str,i16,str,str,str,str
"""tt0000001""",1,"""nm1588970""","""self""",null,"""Self"""
"""tt0000001""",2,"""nm0005690""","""director""",null,null
"""tt0000001""",3,"""nm0005690""","""producer""","""producer""",null
"""tt0000001""",4,"""nm0374658""","""cinematographer""","""director of photography""",null
"""tt0000002""",1,"""nm0721526""","""director""",null,null
…,…,…,…,…,…
"""tt9916880""",17,"""nm0996406""","""director""","""principal director""",null
"""tt9916880""",18,"""nm1482639""","""writer""",null,null
"""tt9916880""",19,"""nm2586970""","""writer""","""books""",null


In [ ]:
title_principals.partition_by("ordering")

## transforming title.crew

In [2]:
test = pl.read_csv(
    "/media/user/Data/dirkv/Code/dagster/dagster-workspace/data/imdb/inputs/imdb_files/title.crew.tsv.gz",
    has_header=True,
    separator="\t",
    truncate_ragged_lines=True,
    null_values="\\N",
    quote_char=None,
    schema={
        "tconst": pl.Utf8,
        "directors": pl.Utf8,
        "writers": pl.Utf8,
    }
)

directors = test.select(
    pl.col("tconst"),
    pl.col("directors").str.split(","),
).explode("directors")
writers = test.select(
    pl.col("tconst"),
    pl.col("writers").str.split(","),
).explode("writers")

In [3]:
directors

tconst,directors
str,str
"""tt0000001""","""nm0005690"""
"""tt0000002""","""nm0721526"""
"""tt0000003""","""nm0721526"""
"""tt0000004""","""nm0721526"""
"""tt0000005""","""nm0005690"""
…,…
"""tt9916848""","""nm1485677"""
"""tt9916850""","""nm1485677"""
"""tt9916852""","""nm1485677"""


## transforming title.ratings (is al goed, alleen rekening houden met \N)

In [3]:
title_ratings = pl.read_csv(
    "/media/user/Data/dirkv/Code/dagster/dagster-workspace/data/imdb/inputs/imdb_files/title.ratings.tsv.gz",
    has_header=True,
    separator="\t",
    truncate_ragged_lines=True,
    null_values="\\N",
    quote_char=None,
    schema={
        "tconst": pl.Utf8,
        "averageRating": pl.Float16,
        "numVotes": pl.UInt32,
    }
)

title_ratings

tconst,averageRating,numVotes
str,f16,u32
"""tt0000001""",5.699219,2200
"""tt0000002""",5.5,311
"""tt0000003""",6.3984375,2308
"""tt0000004""",5.1015625,196
"""tt0000005""",6.199219,3037
…,…,…
"""tt9916846""",5.300781,7
"""tt9916848""",5.199219,7
"""tt9916850""",6.0,7


## title.episode  (is al goed, alleen rekening houden met \N)

In [21]:
test = pl.read_csv(
    "/media/user/Data/dirkv/Code/dagster/dagster-workspace/data/imdb/inputs/imdb_files/title.episode.tsv.gz",
    has_header=True,
    separator="\t",
    truncate_ragged_lines=True,
    null_values="\\N",
    quote_char=None,
    schema={
        "tconst": pl.Utf8,
        "parentTconst": pl.Utf8,
        "seasonNumber": pl.UInt16,
        "episodeNumber": pl.UInt32,
    }
)
test

tconst,parentTconst,seasonNumber,episodeNumber
str,str,u16,u32
"""tt0031458""","""tt32857063""",null,null
"""tt0041951""","""tt0041038""",1,9
"""tt0042816""","""tt0989125""",1,17
"""tt0042889""","""tt0989125""",null,null
"""tt0043426""","""tt0040051""",3,42
…,…,…,…
"""tt9916846""","""tt1289683""",3,18
"""tt9916848""","""tt1289683""",3,17
"""tt9916850""","""tt1289683""",3,19


## title.akas (is al goed, alleen rekening houden met \N en een boolean)

In [ ]:
title_akas = pl.read_csv(
    "/media/user/Data/dirkv/Code/dagster/dagster-workspace/data/imdb/inputs/imdb_files/title.akas.tsv.gz",
    has_header=True,
    separator="\t",
    truncate_ragged_lines=True,
    null_values="\\N",
    quote_char=None,
    schema={
        "titleId": pl.Utf8,
        "ordering": pl.UInt16,
        "title": pl.Utf8,
        "region": pl.Utf8,
        "language": pl.Utf8,
        "types": pl.Utf8,
        "attributes": pl.Utf8,
        "isOriginalTitle": pl.UInt8
    }
)
title_akas = title_akas.with_columns(
        pl.col("isOriginalTitle").cast(pl.Boolean)
    )

ColumnNotFoundError: unable to find column "isAdult"; valid columns: ["titleId", "ordering", "title", "region", "language", "types", "attributes", "isOriginalTitle"]

# kijken of title ratings en titlebasics op elkaar zijn afgestemd

In [25]:
# title_basics.select(
#     pl.col("tconst")
# ).equals(
#     title_ratings.select(
#         pl.col("tconst")
#     )
# )

len(set(title_principals["tconst"]) - set(title_basics["tconst"])) # wat zit in 1e, maar niet in 2e

1816

In [26]:
len(set(title_basics["tconst"]) - set(title_principals["tconst"])) # wat zit in 1e, maar niet in 2e

1181351

In [28]:
set(title_basics["tconst"]) - set(title_principals["tconst"])

{'tt9800348',
 'tt6795662',
 'tt16067678',
 'tt31795391',
 'tt2344119',
 'tt15885712',
 'tt21987146',
 'tt13624372',
 'tt1012401',
 'tt12677626',
 'tt33366230',
 'tt35235652',
 'tt7469836',
 'tt39293201',
 'tt9343670',
 'tt15637982',
 'tt5392096',
 'tt28308445',
 'tt7740882',
 'tt6482262',
 'tt37231547',
 'tt2231448',
 'tt9805200',
 'tt30280951',
 'tt33339076',
 'tt3064108',
 'tt2671902',
 'tt15974646',
 'tt13457254',
 'tt37592744',
 'tt1052230',
 'tt11992096',
 'tt37561072',
 'tt12662136',
 'tt7328562',
 'tt38128298',
 'tt9716046',
 'tt35108065',
 'tt15826788',
 'tt6928736',
 'tt12193894',
 'tt35483706',
 'tt39380246',
 'tt8752648',
 'tt33377777',
 'tt9897220',
 'tt15056772',
 'tt39163564',
 'tt15946210',
 'tt33466071',
 'tt11986484',
 'tt4087736',
 'tt21616652',
 'tt19309582',
 'tt21393900',
 'tt3626682',
 'tt38516794',
 'tt35613590',
 'tt32160034',
 'tt32568855',
 'tt19115000',
 'tt31418728',
 'tt8255336',
 'tt7853080',
 'tt14831904',
 'tt7528622',
 'tt15800624',
 'tt2264628',
 'tt3

In [17]:
title_basics.head()

tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes
str,str,str,str,i64,i64,i64,i64
"""tt0000001""","""short""","""Carmencita""","""Carmencita""",0,1894,null,1
"""tt0000002""","""short""","""Le clown et ses chiens""","""Le clown et ses chiens""",0,1892,null,5
"""tt0000003""","""short""","""Poor Pierrot""","""Pauvre Pierrot""",0,1892,null,5
"""tt0000004""","""short""","""Un bon bock""","""Un bon bock""",0,1892,null,12
"""tt0000005""","""short""","""Blacksmith Scene""","""Blacksmith Scene""",0,1893,null,1


In [18]:
title_principals.head()

tconst,ordering,nconst,category,job,characters
str,i64,str,str,str,str
"""tt0000001""",1,"""nm1588970""","""self""",null,"""Self"""
"""tt0000001""",2,"""nm0005690""","""director""",null,null
"""tt0000001""",3,"""nm0005690""","""producer""","""producer""",null
"""tt0000001""",4,"""nm0374658""","""cinematographer""","""director of photography""",null
"""tt0000002""",1,"""nm0721526""","""director""",null,null


In [32]:
title_basics.select(
    pl.col("runtimeMinutes").max(),
    pl.col("tconst")
    )

runtimeMinutes,tconst
i64,str
3692080,"""tt0000001"""
3692080,"""tt0000002"""
3692080,"""tt0000003"""
3692080,"""tt0000004"""
3692080,"""tt0000005"""
…,…
3692080,"""tt9916848"""
3692080,"""tt9916850"""
3692080,"""tt9916852"""
